# 02 — Visual embedding baselines

**Phase 2 status:** infrastructure smoke test. This notebook exercises the same embedding store and exact-cosine retrieval code used by real DINOv2 / CLIP / SigLIP runs, but it deliberately does **not** download model weights or heritage images during CI.

Full model extraction is an explicit, user-run step because reproducibility should not have a hidden several-gigabyte side effect.

In [ ]:
from pathlib import Path
import sys
import tempfile

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from caypollard.retrieval import top_k_cosine
from caypollard.vision.encoders import DEFAULT_MODELS
from caypollard.embeddings.store import load_embedding_table, save_embedding_table

## Declared baseline encoders

The initial comparison deliberately spans a self-supervised visual model (DINOv2) and vision-language models (CLIP and SigLIP). Release runs must record the resolved immutable model revision.

In [ ]:
for key, spec in DEFAULT_MODELS.items():
    print(f"{key:16s} family={spec.family:7s} model={spec.model_id}")

## Offline storage/retrieval smoke test

These vectors are tiny deterministic fixtures, **not experimental results**. Their purpose is to prove that identifier alignment, normalization, persistence, and exact-cosine ranking are executable without model downloads.

In [ ]:
ids = ["fixture:a", "fixture:b", "fixture:c", "fixture:d"]
vectors = np.array([
    [1.0, 0.0, 0.0],
    [0.8, 0.2, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
], dtype=np.float32)

with tempfile.TemporaryDirectory() as tmpdir:
    out = Path(tmpdir) / "visual_embedding_smoke.npz"
    save_embedding_table(
        out,
        ids=ids,
        vectors=vectors,
        metadata={
            "kind": "deterministic fixture",
            "warning": "not a research result",
        },
    )
    table = load_embedding_table(out)

print(table.vectors.shape, table.ids)

In [ ]:
query_index = 0
results = top_k_cosine(
    table.vectors[query_index],
    table.vectors,
    k=3,
    exclude_index=query_index,
)
[(table.ids[item.index], round(item.score, 4)) for item in results]

## Running a real encoder

After reconstructing the Iconclass image corpus and benchmark manifest:

```bash
uv sync --extra vision

uv run python scripts/embed_images.py \
  data/derived/iconclass-v0.1/manifest.jsonl \
  data/raw/iconclass-testset/images \
  results/embeddings/dinov2-base.npz \
  --preset dinov2-base \
  --batch-size 32
```

Equivalent presets are `clip-vit-b32` and `siglip-base-224`. Each output receives a JSON sidecar containing the manifest checksum, model identifier, requested and resolved Hub revision, processor class/config, library versions, pooling rule, vector dimension, and normalization policy.

## Release gate

A benchmark result is not release-ready unless:

1. the corpus manifest SHA-256 is recorded;
2. the model resolves to an immutable revision rather than merely a mutable model name;
3. image preprocessing is reconstructable from the saved processor configuration;
4. query images are excluded from their own result rankings;
5. all compared encoders use the same split and evaluation definitions.

The next experimental step is to run these encoders on the leakage-controlled Iconclass benchmark and populate the visual-only baseline table.